## Codificar programas faltantes de las bases  de censos universitarios y TUNI

In [22]:
# Instalar librerías
# !pip install -q pandas numpy scikit-learn sentence-transformers torch

In [23]:
# Paquetes
from pathlib import Path
import unicodedata

import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer

In [24]:
# Función de estandarización
def estandarizar_programa(texto):
    if pd.isna(texto):
        return pd.NA

    texto = unicodedata.normalize("NFD", str(texto).upper())

    texto = "".join(
        caracter if caracter.isalnum() else " "
        for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )

    return " ".join(texto.split())

In [25]:
# Carpeta de correspondencias
carpeta = Path(
    r"D:\Daniel_Pajita\1_Proyectos\Research\Educacion"
    r"\educacion-universitaria-peru"
    r"\2_Datos\Correspondencias_entre_clasificaciones"
)

# Entrenamiento
df_train = pd.read_csv(carpeta / "BD_tuni_programas_entrenamiento.txt", sep="|", dtype="string", encoding="utf-8-sig",)

# Programas pendientes de clasificación
archivos_prediccion = [
    "BD_tuni_programas_predicción.txt",
    "programas_censo1996.txt",
    "programas_censo2010.txt",
]

df_pred = pd.concat(
    [
        pd.read_csv(carpeta / archivo, sep="|", dtype="string", encoding="utf-8-sig",)
        for archivo in archivos_prediccion
    ],
    ignore_index=True,
)

In [26]:
# Conservar originales y estandarizar ambas bases
for df in [df_train, df_pred]:
    # Conservar el nombre antes de estandarizar
    df["nombre_programa_original"] = df["nombre_programa"]

    # Estandarizar el nombre
    df["nombre_programa"] = (
        df["nombre_programa"]
        .map(estandarizar_programa).astype("string").replace("", pd.NA)
    )

    # Normalizar el nivel para comparación y texto del modelo
    df["nivel_academico_clean"] = (
        df["nivel_academico"]
        .map(estandarizar_programa).astype("string").replace("", pd.NA).replace({"CARRERA PROFESIONAL": "PREGRADO"})
    )

# Excluir registros que no permiten identificar programa y nivel
campos_requeridos = ["nombre_programa", "nivel_academico_clean"]

for nombre, df in [
    ("Entrenamiento", df_train),
    ("Predicción", df_pred),
]:
    print(
        f"{nombre}: "
        f"{df[campos_requeridos].isna().any(axis=1).sum():,} "
        "filas sin nombre o nivel"
    )

df_train = df_train.dropna(subset=campos_requeridos).copy()
df_pred = df_pred.dropna(subset=campos_requeridos).copy()

Entrenamiento: 0 filas sin nombre o nivel
Predicción: 1 filas sin nombre o nivel


In [28]:
# Revisar conflictos en entrenamiento y eliminar duplicados
claves = ["nivel_academico_clean", "nombre_programa"]

cols_to_copy = [
    "codigo_grupo_1",
    "nombre_grupo_1",
    "codigo_grupo_3",
    "nombre_grupo_3",
]

columnas_codigos = ["codigo_grupo_1", "codigo_grupo_3"]

# Limpiar espacios y reconocer campos vacíos
for columna in cols_to_copy:
    df_train[columna] = (
        df_train[columna]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

if df_train["codigo_grupo_1"].isna().any():
    raise ValueError(
        "Hay filas de entrenamiento sin código de grupo 1."
    )

# Detectar códigos distintos para un mismo programa y nivel
# Un código faltante y uno informado también requieren revisión
conflictos = (df_train.groupby(claves)[columnas_codigos].nunique(dropna=False).gt(1).any(axis=1))

claves_conflictivas = (conflictos[conflictos].reset_index()[claves])

# Identificar las filas correspondientes a claves conflictivas
indice_entrenamiento = pd.MultiIndex.from_frame(df_train[claves])
indice_conflictos = pd.MultiIndex.from_frame(claves_conflictivas)

es_conflictivo = indice_entrenamiento.isin(indice_conflictos)

# Conservar los conflictos para revisión
conflictos_entrenamiento = (df_train.loc[es_conflictivo].copy().sort_values(claves).reset_index(drop=True))

if not conflictos_entrenamiento.empty:
    print(
        "Combinaciones de programa y nivel con conflictos:",
        f"{len(claves_conflictivas):,}",
    )
    print(
        "Filas excluidas del entrenamiento:",
        f"{len(conflictos_entrenamiento):,}",
    )

    with pd.option_context(
        "display.max_rows", None,
        "display.max_colwidth", None,
    ):
        display(
            conflictos_entrenamiento[
                claves
                + ["nombre_programa_original"]
                + cols_to_copy
            ]
        )
else:
    print("No se detectaron conflictos en entrenamiento.")

# Continuar únicamente con referencias sin conflictos
df_train = df_train.loc[~es_conflictivo].copy()

# Con códigos iguales, priorizar las etiquetas más completas
df_train = (
    df_train
    .assign(
        _completitud=lambda df: df[cols_to_copy].notna().sum(axis=1)
    )
    .sort_values(
        "_completitud",
        ascending=False,
        kind="stable",
    )
    .drop_duplicates(subset=claves)
    .drop(columns="_completitud").reset_index(drop=True)
)

# Predicción: una fila por nombre estandarizado y nivel
df_pred = (df_pred.drop_duplicates(subset=claves).reset_index(drop=True))

if df_train.empty or df_pred.empty:
    raise ValueError(
        "Entrenamiento o predicción quedó sin filas."
    )

assert not df_train.duplicated(claves).any()
assert not df_pred.duplicated(claves).any()

print(f"Entrenamiento disponible: {len(df_train):,} programas")
print(f"Predicción: {len(df_pred):,} programas")

Combinaciones de programa y nivel con conflictos: 5
Filas excluidas del entrenamiento: 10


,nivel_academico_clean,nombre_programa,nombre_programa_original,codigo_grupo_1,nombre_grupo_1,codigo_grupo_3,nombre_grupo_3
0,MAESTRIA,MAESTRIA EN DERECHO CON MENCION EN DERECHO CIVIL Y COMERCIAL,Maestría en Derecho con Mención en Derecho Civil y Comercial,4,Ciencias Administrativas y Derecho,422,Derecho Privado
1,MAESTRIA,MAESTRIA EN DERECHO CON MENCION EN DERECHO CIVIL Y COMERCIAL,Maestría en Derecho con Mención en Derecho Civil y Comercial,4,Ciencias Administrativas y Derecho,421,Derecho
2,MAESTRIA,MAESTRIA EN EDUCACION CON MENCION EN GESTION EDUCATIVA,Maestría en Educación con Mención en Gestión Educativa,1,Educación,191,Gestión y Calidad Educativa
3,MAESTRIA,MAESTRIA EN EDUCACION CON MENCION EN GESTION EDUCATIVA,"Maestría en Educación, con Mención en Gestión Educativa",1,Educación,141,Gestión y Calidad Educativa
4,PREGRADO,EDUCACION ARTISTICA ESPECIALIDAD MUSICA,Educación Artística Especialidad Música,1,Educación,123,Educación Artística
5,PREGRADO,EDUCACION ARTISTICA ESPECIALIDAD MUSICA,"Educación Artística, Especialidad Música",1,Educación,161,Educación Artística
6,PREGRADO,EDUCACION SECUNDARIA ESPECIALIDAD CIENCIAS SOCIALES,Educación Secundaria especialidad Ciencias Sociales,1,Educación,121,Educación Secundaria
7,PREGRADO,EDUCACION SECUNDARIA ESPECIALIDAD CIENCIAS SOCIALES,"Educación Secundaria, especialidad: Ciencias Sociales",1,Educación,113,Educación Secundaria
8,PREGRADO,EDUCACION SECUNDARIA ESPECIALIDAD INGLES,Educación Secundaria Especialidad: Inglés,1,Educación,121,Educación Secundaria
9,PREGRADO,EDUCACION SECUNDARIA ESPECIALIDAD INGLES,"Educación Secundaria, especialidad: Inglés",1,Educación,113,Educación Secundaria


Entrenamiento disponible: 5,085 programas
Predicción: 12,964 programas


In [29]:
# Construir el texto del modelo
for df in [df_train, df_pred]:
    df["texto_modelo"] = (df["nivel_academico_clean"].str.lower() + " | " + df["nombre_programa"].str.lower())

In [30]:
# Cargar el modelo
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [31]:
# Crear los embeddings
emb_train = model.encode(df_train["texto_modelo"].tolist(),show_progress_bar=True,normalize_embeddings=True,)
emb_pred = model.encode(df_pred["texto_modelo"].tolist(),show_progress_bar=True,normalize_embeddings=True,)

Batches:   0%|          | 0/159 [00:00<?, ?it/s]

Batches:   0%|          | 0/406 [00:00<?, ?it/s]

In [32]:
# Buscar el vecino más cercano
nn = NearestNeighbors(n_neighbors=1,metric="cosine",)
nn.fit(emb_train)
dist, idx = nn.kneighbors(emb_pred)
df_pred["match_score"] = 1 - dist.flatten()
df_pred["match_index"] = idx.flatten()

In [33]:
# Incorporar los grupos del programa de referencia
referencias = (df_train.iloc[df_pred["match_index"].to_numpy()].reset_index(drop=True))

for columna in cols_to_copy:
    df_pred[columna] = referencias[columna].to_numpy()

df_pred["nombre_programa_referencia"] = (referencias["nombre_programa"].to_numpy())
df_pred["nivel_academico_referencia"] = (referencias["nivel_academico"].to_numpy())

In [34]:
# Marcar casos para revisión
umbral = 0.75
df_pred["revision_manual"] = (df_pred["match_score"] < umbral).astype(int)

columnas_revision = [
    "nivel_academico",
    "nombre_programa_original",
    "nombre_programa",
    "nombre_programa_referencia",
    "nivel_academico_referencia",
    "match_score",
] + cols_to_copy

display(df_pred.loc[df_pred["revision_manual"].eq(1), columnas_revision,].sort_values("match_score").head(100))
print("Programas marcados para revisión:", f"{df_pred['revision_manual'].sum():,}",)

,nivel_academico,nombre_programa_original,nombre_programa,nombre_programa_referencia,nivel_academico_referencia,match_score,codigo_grupo_1,nombre_grupo_1,codigo_grupo_3,nombre_grupo_3
5789,Pregrado,AULA 403,AULA 403,EDUCACION CON MENCION EN INICIAL VIRTUAL SEMIP...,Pregrado,0.484212,1,Educación,111,Educación Inicial
559,Pregrado,2311ERA CODIGO PROVISIONAL,2311ERA CODIGO PROVISIONAL,GRABADO,Pregrado,0.512850,2,Arte y Humanidades,213,Arte
558,Pregrado,2295ERA CODIGO PROVISIONAL,2295ERA CODIGO PROVISIONAL,GRABADO,Pregrado,0.518475,2,Arte y Humanidades,213,Arte
557,Pregrado,2273ERA CODIGO PROVISIONAL,2273ERA CODIGO PROVISIONAL,GRABADO,Pregrado,0.519292,2,Arte y Humanidades,213,Arte
10154,Pregrado,NINGUNO,NINGUNO,QUIMICA,Pregrado,0.520038,5,"Ciencias Naturales, Matemáticas y Estadística",531,Química
...,...,...,...,...,...,...,...,...,...,...
8057,Pregrado,EXPORTACION DE PRODUCTOS PARA EL EXTRANGERO,EXPORTACION DE PRODUCTOS PARA EL EXTRANGERO,COMERCIO EXTERIOR Y MARKETING INTERNACIONAL,Pregrado,0.653846,4,Ciencias Administrativas y Derecho,416,Negocios Internacionales
10470,Pregrado,PROGRAMA DE ESTUDIOS COMPLEMENTARIOS PARA CONT...,PROGRAMA DE ESTUDIOS COMPLEMENTARIOS PARA CONT...,MAESTRIA EN DIRECCION COMERCIAL,Maestría,0.654191,4,Ciencias Administrativas y Derecho,414,Marketing y Publicidad
4975,Pregrado,A DISTANCIA,A DISTANCIA,ESTADISTICA,Pregrado,0.654837,5,"Ciencias Naturales, Matemáticas y Estadística",542,Estadística
10521,Pregrado,PROGRAMA ESPECIAL PARA CONTADORES MERCANTILES,PROGRAMA ESPECIAL PARA CONTADORES MERCANTILES,MARKETING,Pregrado,0.654905,4,Ciencias Administrativas y Derecho,414,Marketing y Publicidad


Programas marcados para revisión: 428


In [35]:
# Ordenar y exportar
df_pred = (df_pred.sort_values(["nivel_academico", "nombre_programa"],kind="stable",).reset_index(drop=True))
assert not df_pred.duplicated(claves).any()

salida = carpeta / "BD_programas_prediccion_clasificada.txt"
df_pred.to_csv(salida,sep="|", index=False, encoding="utf-8-sig",)

print(f"Programas exportados: {len(df_pred):,}")
print(f"Archivo: {salida}")

Programas exportados: 12,964
Archivo: D:\Daniel_Pajita\1_Proyectos\Research\Educacion\educacion-universitaria-peru\2_Datos\Correspondencias_entre_clasificaciones\BD_programas_prediccion_clasificada.txt


In [ ]:
# Cargar entrenamiento
carpeta = Path(
    r"D:\Daniel_Pajita\1_Proyectos\Research\Educacion\educacion-universitaria-peru\2_Datos\Correspondencias_entre_clasificaciones"
)

df_train = pd.read_csv(carpeta / "BD_tuni_programas_entrenamiento.txt",sep="|",dtype=str,encoding="utf-8-sig",)

# Combinar los programas pendientes de clasificación
archivos_prediccion = ["BD_tuni_programas_predicción.txt", "programas_censo1996.txt", "programas_censo2010.txt",]

df_pred = pd.concat(
    [
        pd.read_csv(carpeta / archivo,sep="|",dtype=str,encoding="utf-8-sig",)
        for archivo in archivos_prediccion
    ],
    ignore_index=True,
)
df_pred = (df_pred.drop_duplicates(subset=["nivel_academico", "nombre_programa"]).reset_index(drop=True))

print(f"Entrenamiento: {len(df_train):,} programas")
print(f"Predicción: {len(df_pred):,} programas")

Entrenamiento: 5,122 programas
Predicción: 13,000 programas


In [ ]:
# Limpieza 
def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

for df in [df_train, df_pred]:
    df["nivel_academico_clean"] = df["nivel_academico"].apply(clean_text)
    df["nombre_programa_clean"] = df["nombre_programa"].apply(clean_text)

# Texto final que usará el modelo (concatena nivel + nombre)
df_train["texto_modelo"] = df_train["nivel_academico_clean"] + " | " + df_train["nombre_programa_clean"]
df_pred["texto_modelo"]  = df_pred["nivel_academico_clean"]  + " | " + df_pred["nombre_programa_clean"]

In [ ]:
# Modelo multilingüe MiniLM para embeddings
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# Crear embeddings del entrenamiento (solo una vez)
X_train = df_train["texto_modelo"].tolist()
emb_train = model.encode(X_train, show_progress_bar=True, normalize_embeddings=True)

Batches:   0%|          | 0/161 [00:00<?, ?it/s]

In [ ]:
# Clasificar por similitud (KNN por vecinos más cercanos)
nn = NearestNeighbors(n_neighbors=1, metric="cosine")
nn.fit(emb_train)

X_pred = df_pred["texto_modelo"].tolist()
emb_pred = model.encode(X_pred, show_progress_bar=True, normalize_embeddings=True)
dist, idx = nn.kneighbors(emb_pred)  
df_pred["match_score"] = 1 - dist.flatten() 
df_pred["match_index"] = idx.flatten()

Batches:   0%|          | 0/407 [00:00<?, ?it/s]

In [ ]:
# Completar las columnas desde el match
cols_to_copy = ["codigo_grupo_1", "nombre_grupo_1", "codigo_grupo_3", "nombre_grupo_3"]
for c in cols_to_copy:
    df_pred[c] = df_train.iloc[df_pred["match_index"]][c].values

In [ ]:
# Marcar para revisión manual
umbral = 0.95
df_pred["revision_manual"] = (df_pred["match_score"] < umbral).astype(int)
df_pred[df_pred["revision_manual"]==1][["nivel_academico","nombre_programa","match_score"] + cols_to_copy].head(100)

,nivel_academico,nombre_programa,match_score,codigo_grupo_1,nombre_grupo_1,codigo_grupo_3,nombre_grupo_3
1,Pregrado,"Administración Hotelera, Turismo, Ecoturismo y...",0.949243,0,Servicios,12,"Hotelería, Restaurantes y Gastronomía"
2,Pregrado,Administración de Redes y Seguridad Informática,0.815914,6,Tecnología de la Información y la Comunicación,612,Sistemas y Cómputo
5,Pregrado,Administración y Negocios Internacionales Moda...,0.743357,4,Ciencias Administrativas y Derecho,416,Negocios Internacionales
23,Pregrado,Carrera Profesional de Ciencias Administrativa...,0.834999,4,Ciencias Administrativas y Derecho,413,Gestión y Administración
24,Pregrado,Carrera Profesional de Ciencias de la Educació...,0.892075,1,Educación,112,Educación Primaria
...,...,...,...,...,...,...,...
231,Maestría,Máster en Dirección de Empresas A Tiempo Parci...,0.780302,4,Ciencias Administrativas y Derecho,419,Otros Programas en Administración y Comercio
232,Pregrado,Negocios Globales,0.933447,4,Ciencias Administrativas y Derecho,416,Negocios Internacionales
233,Pregrado,Programa De Complementación Pedagógica (PCP),0.848262,1,Educación,199,Otros Programas en Educación
234,Pregrado,Programa De Complementación Pedagógica Y Unive...,0.926979,1,Educación,199,Otros Programas en Educación


In [ ]:
# Estandarizar nombres de programas
def estandarizar_programa(texto):
    if pd.isna(texto):
        return pd.NA

    texto = unicodedata.normalize("NFD", str(texto).upper())
    texto = "".join(
        c if c.isalnum() else " "
        for c in texto
        if unicodedata.category(c) != "Mn"
    )
    return " ".join(texto.split())


# Conservar el nombre original y estandarizar el campo de salida
df_pred["nombre_programa_original"] = df_pred["nombre_programa"]
df_pred["nombre_programa"] = (df_pred["nombre_programa"].map(estandarizar_programa))

In [ ]:
# Número de repeticiones adicionales por programa y nivel
df_pred["dup"] = (df_pred.groupby(["nombre_programa", "nivel_academico"], dropna=False,)["nombre_programa"].transform("size").sub(1))

# Conservar la clasificación con mayor similitud
df_pred["match_score"] = pd.to_numeric(df_pred["match_score"], errors="coerce")
df_pred = (df_pred.sort_values(
        ["nivel_academico", "nombre_programa", "match_score"],ascending=[True, True, False],na_position="last",kind="stable",
    )
    .drop_duplicates(subset=["nivel_academico", "nombre_programa"],keep="first",)
    .reset_index(drop=True)
)
df_pred = df_pred.drop(columns="dup")

In [ ]:
# Guardar
salida = carpeta / "BD_programas_prediccion_clasificada.txt"
df_pred.to_csv(salida, sep="|", index=False, encoding="utf-8")